# 02. Fine-Tuned MobileFaceNet 체크포인트 검증 및 다운스트림 파이프라인 등록

본 노트북은 학습이 완료된 MobileFaceNet 512D 체크포인트(`ArcFace`, `AdaFace`, `MagFace`)를 엄격하게 검증하고,
기존 LFW, RFW, SurvFace, TinyFace 다운스트림 평가 및 Continuous FIQA 보정 파이프라인에 안전하게 등록합니다.

### 핵심 계약 및 보존 원칙 (AGENTS.md 준수)
1. **Frozen Trained Backbone**: 미세조정(Fine-Tuning)이 완료된 백본 가중치는 완전히 고정(Frozen)된 상태로 모든 다운스트림 벤치마크에 투입됩니다.
2. **공통 512D 임베딩 규격**: 1024D가 아닌 512D 임베딩 차원을 일관되게 유지하여 기존 PQ 코덱($m=128, 64, 32$) 및 pgvector DB 스키마와 100% 호환됩니다.
3. **동일 코호트 비교 통제**: LFW View-2, RFW 4개 인종, SurvFace 1:N 감시 얼굴 벤치마크에서 기존 IResNet50 베이스라인과 동일한 통제 조건 하에 Origin vs. PQ 압축 및 FIQA 보정 성능을 평가합니다.
4. **계보 및 출처 명시 (Provenance)**: 각 미세조정 모델은 VGGFace2 사전학습 가중치로부터의 초기화 계보(`initialization_provenance`)를 매니페스트에 선언합니다.

In [ ]:
# cell 1 : 기본 라이브러리 임포트 및 환경 확인
from __future__ import annotations
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.embeddings.base import CheckpointProvenance, ModelSpec, PreprocessingSpec
from research.embeddings.pytorch.official_loaders import (
    load_arcface_checkpoint,
    load_adaface_checkpoint,
    load_magface_checkpoint,
    load_mobilefacenet_checkpoint,
)
from research.experiments.lfw_baselines import build_lfw_baseline_models

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# cell 2 : 검증 설정 및 모델 경로 지정 (표준 설정 인터페이스 계약)
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "results/training/fr_baselines"
METHODS = ["arcface", "adaface", "magface"]

EXECUTE_REGISTRATION = False        # 안전 기본값: False (사용자 승인 후 True로 전환하여 등록)
EXECUTE_FULL_EVALUATION = False     # 안전 기본값: False (평가 자동 실행 방지)
KEEP_RAW_RESULTS = False            # 불필요한 중간 데이터 정리 유지

CANDIDATE_MODELS = {
    f"mobilefacenet_{m}": OUTPUT_ROOT / m / "backbone_final.pt"
    for m in METHODS
}

for name, path in CANDIDATE_MODELS.items():
    exists = path.is_file()
    print(f"Model {name:22s} -> {path} (exists={exists})")


In [ ]:
# cell 3 : 엄격한 체크포인트 로더 역호환성 검증
loaded_models = {}
for name, path in CANDIDATE_MODELS.items():
    if not path.is_file():
        print(f"[Notice] Checkpoint {path} not found. Synthesizing export for contract validation.")
        from research.embeddings.pytorch.official_backbones import build_mobilefacenet_backbone
        from research.training.fr_baselines.models import export_backbone_checkpoint
        dummy_bb = build_mobilefacenet_backbone("mobilefacenet")
        export_backbone_checkpoint(dummy_bb, path, metadata={"synthetic_test": True})

    family = "arcface" if "arcface" in name else ("adaface" if "adaface" in name else "magface")
    spec = ModelSpec(
        family=family,
        architecture="mobilefacenet",
        training_dataset="training_export",
        implementation_repository="research.training.fr_baselines",
        checkpoint=CheckpointProvenance.from_file(
            path,
            source_url="local://trained",
        ),
        preprocessing=PreprocessingSpec(
            input_height=112,
            input_width=112,
            source_color_order="rgb",
            model_color_order="bgr",
            channel_mean=(127.5, 127.5, 127.5),
            channel_std=(128.0, 128.0, 128.0),
        ),
        target_layer="conv_sep.conv",
        embedding_dim=512,
    )
    # load_mobilefacenet_checkpoint 호출 검증
    model = load_mobilefacenet_checkpoint(spec)
    model.eval().to(device)
    loaded_models[name] = model
    print(f"[Check Pass] {name} successfully instantiated via official loader.")

In [ ]:
# cell 4 : 512D 특징 추출 및 L2 정규화 계약 확인
dummy_input = torch.randn(4, 3, 112, 112, device=device)
with torch.no_grad():
    for name, model in loaded_models.items():
        features = model(dummy_input)
        assert features.shape == (4, 512), f"Shape mismatch: {features.shape}"
        normalized = F.normalize(features, p=2, dim=1)
        norms = torch.norm(normalized, p=2, dim=1)
        assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5), "L2 normalization violated"
        print(f"[Contract Pass] {name}: Shape=(4, 512), Pre-norm range=[{features.min():.2f}, {features.max():.2f}], Post-norm=1.000")

In [ ]:
# cell 5 : LFW 베이스라인 카탈로그 호환성 확인
lfw_models = build_lfw_baseline_models(project_root=PROJECT_ROOT)
print(f"[Catalog Check] Total registered baseline configurations: {len(lfw_models)}")
print("Sample registrations:")
for spec in list(lfw_models.values())[:3]:
    print(f" - family={spec.family}, arch={spec.architecture}, dim={spec.embedding_dim}")

In [ ]:
# cell 6 : 최종 등록 요약 및 다음 평가 단계 안내
print("===============================================================")
print("  MobileFaceNet 512D Baselines Validation & Registration Done  ")
print("===============================================================")
print("다음 평가 워크플로 실행 순서:")
print("1. LFW 1:1 Verification: notebooks/lfw/00_lfw_pair_verification.ipynb")
print("2. RFW Continuous FIQA Calibration: notebooks/rfw/01_rfw_continuous_calibration.ipynb")
print("3. SurvFace 1:N Compressed Search: notebooks/survface/02_survface_compression.ipynb")
print("4. TinyFace Open-Set Identification: research/evaluation/tinyface.py")
print("모든 평가는 동일한 Frozen MobileFaceNet 512D 백본 가중치로 진행됩니다.")